# Spectral Anomaly Detection (SAD)

This notebook demonstrates the Spectral Anomaly Detection algorithm for
identifying radioactive sources in gamma-ray time series data.

## Algorithm Overview

The SAD detector:
1. **Learns a background subspace** using PCA on source-absent training data
2. **Computes reconstruction error** for new spectra: SAD(x) = ||(I − UUᵀ)x||²
3. **Detects anomalies** when reconstruction error exceeds threshold
4. **Aggregates alarms** that are close in time

## Reference

Miller, K., & Dubrawski, A. (2018). Gamma-ray source detection with small
sensors. *IEEE Transactions on Nuclear Science*, 65(4), 1047–1058.

## Dataset

Using the TopCoder Urban Data Challenge dataset (mobile NaI detector).

**Note:** You must download the TopCoder dataset separately and set
`TOPCODER_DATA_DIR` below to point to it.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from gammaflow import SpectralTimeSeries
from gammaflow.core.spectra import Spectra
from gammaflow.algorithms import SADDetector
from gammaflow.datasets import TopCoderDataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

# Path to the TopCoder dataset on your machine
TOPCODER_DATA_DIR = 'PATH/TO/topcoder'

ds = TopCoderDataset(TOPCODER_DATA_DIR)
print(ds)

## Load Background Training Data

SAD needs background-only data to learn the "normal" spectral subspace via PCA.

In [ ]:
background_run_ids = ds.list_runs(source_id=0)
print(f"Found {len(background_run_ids)} background runs")

background_spectra_list = []
total_spectra = 0

for run_id in tqdm(background_run_ids, desc='Loading background'):
    try:
        listmode, _ = ds.load_run(run_id)
        ts = SpectralTimeSeries.from_list_mode(
            listmode,
            integration_time=5.0,
            stride_time=5.0,
            energy_bins=512,
            energy_range=(0, 3000),
        )
        background_spectra_list.append(ts)
        total_spectra += ts.n_spectra
    except Exception as e:
        print(f"  Warning: run {run_id} failed: {e}")

all_bg_spectra = []
for ts in background_spectra_list:
    all_bg_spectra.extend(ts.spectra)

background_training = Spectra(all_bg_spectra)
print(f"Training set: {len(background_training.spectra)} spectra "
      f"from {len(background_spectra_list)} runs")

## Train SAD Detector

We use `normalize=False` to preserve count rate information.  With
normalization, total count rate changes are erased and only spectral
shape matters, which may hurt performance.

In [ ]:
detector = SADDetector(
    n_components=5,
    normalize=False,
    aggregation_gap=2.0,
)

detector.fit(background_training)

var_ratios = detector.get_explained_variance_ratio()
cum_var = detector.get_cumulative_variance_explained()

print("Explained variance by component:")
for i, v in enumerate(var_ratios, 1):
    print(f"  PC{i}: {v*100:.2f}%")
print(f"Cumulative: {cum_var*100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.bar(range(1, len(var_ratios)+1), var_ratios*100, color='steelblue',
        edgecolor='black')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('Individual Explained Variance', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(var_ratios)+1), np.cumsum(var_ratios)*100,
         'o-', lw=2, ms=8, color='steelblue')
ax2.axhline(95, color='red', ls='--', lw=1, alpha=0.5, label='95%')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance (%)')
ax2.set_title('Cumulative Explained Variance', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Set Threshold Based on False Alarm Rate

We calibrate the threshold using the **alarms per hour** metric
(ANSI N42 standards typically require < 1 alarm/hour at the high end
down to < 0.125 alarms/hour at the low end, depending on CONOPS).
We concatenate background runs to build at least 1 hour of calibration
data for reliable FAR estimation.

In [ ]:
alarms_per_hour = 1
MIN_CALIBRATION_HOURS = 1.0

# Build a calibration dataset from multiple background runs (>= 1 hour)
cal_spectra = []
cal_total_time = 0.0
for ts in background_spectra_list:
    cal_spectra.extend(ts.spectra)
    cal_total_time += sum(s.real_time for s in ts.spectra)
    if cal_total_time >= MIN_CALIBRATION_HOURS * 3600.0:
        break

cal_counts = np.array([s.counts for s in cal_spectra])
cal_real_times = np.array([s.real_time for s in cal_spectra])
cal_live_times = np.array([
    s.live_time if s.live_time is not None else s.real_time
    for s in cal_spectra
])
cal_timestamps = np.cumsum(cal_real_times)
cal_energy_edges = background_spectra_list[0].spectra[0].energy_edges

cal_time_series = SpectralTimeSeries.from_array(
    cal_counts,
    energy_edges=cal_energy_edges,
    timestamps=cal_timestamps,
    real_times=cal_real_times,
    live_times=cal_live_times,
)

print(f"Calibration dataset: {len(cal_spectra)} spectra, "
      f"{cal_total_time/3600:.2f} hours")

detector.set_threshold_by_far(
    cal_time_series,
    alarms_per_hour=alarms_per_hour,
    verbose=True,
)

print(f"\nThreshold: {detector.threshold:.6f}")

# Score distribution on calibration data
all_scores = detector.score_time_series(cal_time_series)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(all_scores, bins=50, alpha=0.7, edgecolor='black')
ax1.axvline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold = {detector.threshold:.6f}')
ax1.set_xlabel('SAD Score')
ax1.set_ylabel('Count')
ax1.set_title('SAD Score Distribution (Background)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

sorted_scores = np.sort(all_scores)
cdf = np.arange(1, len(sorted_scores)+1) / len(sorted_scores)
ax2.plot(sorted_scores, cdf, lw=2)
ax2.axvline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold = {detector.threshold:.6f}')
ax2.set_xlabel('SAD Score')
ax2.set_ylabel('Cumulative Probability')
ax2.set_title('CDF', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Test on Run with Source

In [ ]:
# Pick the strongest I-131 run
ak = ds.get_answer_key('training')
i131_runs = ak[ak['SourceID'] == 3].sort_values('Speed/Offset', ascending=False)
test_run_id = int(i131_runs.iloc[0]['RunID'])

listmode, metadata = ds.load_run(test_run_id)

print(f"Run {test_run_id}")
print(f"  Source: {metadata['SourceName']}")
print(f"  Source Time: {metadata['SourceTime']:.1f} s")
print(f"  Speed/Offset: {metadata['Speed/Offset']:.2f}")

test_ts = SpectralTimeSeries.from_list_mode(
    listmode,
    integration_time=1.0,
    stride_time=1.0,
    energy_bins=512,
    energy_range=(0, 3000),
)
print(f"  {test_ts}")

## Run SAD Detection

In [ ]:
sad_scores = detector.process_time_series(test_ts)
summary = detector.get_alarm_summary()

print(f"Alarms: {summary['n_alarms']}")
if summary['n_alarms'] > 0:
    print(f"Peak SAD score: {summary['max_peak_metric']:.6f}")
    print(f"Total alarm time: {summary['total_alarm_time']:.2f} s")

true_source_time = metadata['SourceTime']
for i, alarm in enumerate(detector.alarms, 1):
    print(f"  {i}. {alarm}")
    if alarm.start_time <= true_source_time <= alarm.end_time:
        print(f"      Captured true source (t={true_source_time:.1f}s)")

## Visualize Results

In [ ]:
times = test_ts.timestamps
count_rates = np.array([
    float(s.counts.sum()) / float(
        s.live_time if s.live_time is not None else s.real_time
    )
    for s in test_ts.spectra
])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Count rate
ax1.step(times, count_rates, where='post', color='black', lw=1.5,
         label='Count rate')
ax1.set_ylabel(r'Count Rate (s$^{-1}$)', fontsize=12)
ax1.set_title(
    f'SAD Detection - Run {test_run_id} ({metadata["SourceName"]})',
    fontsize=14, fontweight='bold',
)
ax1.grid(True, alpha=0.3)
for i, alarm in enumerate(detector.alarms):
    ax1.axvspan(alarm.start_time, alarm.end_time, alpha=0.3, color='red',
                label='Alarm' if i == 0 else '')
if metadata['SourceID'] != 0:
    ax1.axvline(metadata['SourceTime'], color='green', ls='--', lw=2,
                label='True source')
ax1.legend(loc='best')

# SAD scores
ax2.step(times, sad_scores, where='post', color='steelblue', lw=1.5,
         label='SAD score')
ax2.axhline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold ({alarms_per_hour} alarms/hr)')
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('SAD Score (Reconstruction Error)', fontsize=12)
ax2.set_title('Spectral Anomaly Metric', fontsize=13, fontweight='bold')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)
for alarm in detector.alarms:
    ax2.axvspan(alarm.start_time, alarm.end_time, alpha=0.2, color='red')
ax2.legend(loc='best')

plt.tight_layout()
plt.show()